# 03 — Clustering

Cluster wallets into behavioral archetypes, validate statistically and on Etherscan, save final assignments.

**Inputs:**  `data/processed/wallet_features_scaled.parquet`, `wallet_features.parquet`
**Outputs:** `data/processed/clusters.parquet`, `data/processed/models/kmeans_kN.joblib`, `data/processed/umap_emb.npy`, `docs/umap_clusters.png`, `docs/pca_clusters.png`, `docs/cluster_profiles.png`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.decomposition import PCA

from src.models.clustering import sweep_kmeans, fit_kmeans, fit_isolation_forest
from src.models.evaluation import cluster_profiles, cluster_sizes, umap_embed, top_wallets_per_cluster

PROCESSED = Path("../data/processed")
MODELS = PROCESSED / "models"
MODELS.mkdir(exist_ok=True)
DOCS = Path("../docs")
DOCS.mkdir(exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100

ImportError: cannot import name 'sweep_kmeans' from 'src.models' (/Users/ck/Documents/GitHub/ucsd_chainsense/src/models/__init__.py)

## 0. Load data

If wallet count is much above ~50k, consider raising `MIN_ACTIVITY` in `src/features/wallet_features.py` and rebuilding. Large counts dilute clusters with low-activity noise and make every step slower.

In [ ]:
scaled   = pd.read_parquet(PROCESSED / "wallet_features_scaled.parquet")
features = pd.read_parquet(PROCESSED / "wallet_features.parquet")

FEATURE_COLS = [c for c in scaled.columns if c != "wallet"]
X = scaled[FEATURE_COLS].values
wallets = scaled["wallet"].values

print(f"Wallets:  {len(scaled):,}")
print(f"Features: {len(FEATURE_COLS)}")

if len(scaled) > 50_000:
    print(f"\nWARN: {len(scaled):,} wallets is on the high side.")
    print("Consider raising MIN_ACTIVITY in wallet_features.py to get cleaner clusters.")

## 1. K sweep

Find the right number of clusters. Silhouette is the most reliable metric; inertia gives the classic elbow plot; Davies-Bouldin is a sanity check.

Pick k where silhouette peaks. If silhouette is flat across all k, your features don't separate well and no value of k will fix it — go back to feature engineering.

In [ ]:
sweep = sweep_kmeans(X, k_range=range(2, 13))
sweep

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(sweep["k"], sweep["silhouette"], "o-")
axes[0].set_xlabel("k"); axes[0].set_ylabel("silhouette")
axes[0].set_title("Silhouette (higher is better)")

axes[1].plot(sweep["k"], sweep["inertia"], "o-")
axes[1].set_xlabel("k"); axes[1].set_ylabel("inertia")
axes[1].set_title("Inertia (look for elbow)")

axes[2].plot(sweep["k"], sweep["davies_bouldin"], "o-")
axes[2].set_xlabel("k"); axes[2].set_ylabel("Davies-Bouldin")
axes[2].set_title("Davies-Bouldin (lower is better)")

plt.tight_layout()
plt.show()

## 2. Fit K-Means

Pick `K` based on the sweep above. Re-run from this cell down if you change it.

In [ ]:
K = 6  # <-- update based on sweep

model, labels = fit_kmeans(X, K)
joblib.dump(model, MODELS / f"kmeans_k{K}.joblib")
print(f"Fitted K-Means with k={K}")

In [ ]:
sizes = cluster_sizes(labels)
print(sizes)
print()

biggest_share = sizes.max() / len(labels)
if biggest_share > 0.7:
    print(f"WARN: one cluster holds {biggest_share:.0%} of wallets.")
    print("Features aren't separating well — try log-transforming more columns or raising MIN_ACTIVITY.")

fig, ax = plt.subplots(figsize=(8, 4))
sizes.plot(kind="bar", ax=ax)
ax.set_xlabel("cluster"); ax.set_ylabel("# wallets")
ax.set_title(f"Cluster sizes (k={K})")
plt.tight_layout()
plt.show()

## 3. Cluster profiles

The interpretation step. Each cluster's z-score signature tells you what makes it different from the global average. Read each row — strong positive/negative cells are the cluster's identity.

In [ ]:
profiles = cluster_profiles(scaled, labels)
profiles.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 1 + 0.5 * K))
sns.heatmap(
    profiles, center=0, cmap="RdBu_r", annot=True, fmt=".1f",
    cbar_kws={"label": "z-score vs global mean"}, ax=ax,
)
ax.set_title("Cluster signatures")
ax.set_xlabel("feature"); ax.set_ylabel("cluster")
plt.tight_layout()
plt.savefig(DOCS / "cluster_profiles.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Assign archetype labels

Based on the heatmap, name each cluster. The reasoning in comments is what you'll port to the README.

Common patterns:
- High `tx_count` + high `failed_tx_ratio` + low `total_value_eth_sent` → **bot / sniper**
- High `total_value_eth_sent` + low `tx_count` → **whale**
- High `contract_call_ratio` + many `unique_recipients` → **trader / DEX user**
- High `tokens_received_count` + low send activity → **airdrop farmer**
- Mid activity across the board → **casual user**

In [ ]:
ARCHETYPES = {
    0: "cluster_0_name",   # describe based on profile row 0
    1: "cluster_1_name",
    2: "cluster_2_name",
    3: "cluster_3_name",
    4: "cluster_4_name",
    5: "cluster_5_name",
}

assert set(ARCHETYPES.keys()) == set(np.unique(labels)), "ARCHETYPES keys must match cluster labels"
print("Archetype map valid.")

## 5. PCA visualization (fast)

Linear projection — if clusters separate well here, the structure is simple and describable. Always cheap to run; do this first.

In [ ]:
pca_emb = PCA(n_components=2, random_state=42).fit_transform(X)

fig, ax = plt.subplots(figsize=(10, 8))
palette = sns.color_palette("tab10", n_colors=K)
for c in sorted(set(labels)):
    mask = labels == c
    ax.scatter(
        pca_emb[mask, 0], pca_emb[mask, 1],
        s=6, alpha=0.4, color=palette[c],
        label=f"{c}: {ARCHETYPES[c]}",
    )

ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.set_title(f"K-Means clusters in PCA space (k={K})")
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig(DOCS / "pca_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. UMAP visualization (slow, cached)

Nonlinear projection. Better at showing cluster geometry than PCA, but slow.

The cell below caches `umap_emb.npy` so subsequent runs are instant. Delete that file to recompute.

At 50k+ wallets, full UMAP takes minutes. If it's too slow, skip this cell and use the subsample fallback below.

In [ ]:
UMAP_CACHE = PROCESSED / "umap_emb.npy"

if UMAP_CACHE.exists():
    emb = np.load(UMAP_CACHE)
    print(f"Loaded cached UMAP: {emb.shape}")
    if len(emb) != len(X):
        print(f"WARN: cache size {len(emb):,} != X size {len(X):,}. Delete cache and re-run.")
else:
    print(f"Computing UMAP on {len(X):,} wallets — this may take several minutes...")
    emb = umap_embed(X)
    np.save(UMAP_CACHE, emb)
    print(f"Saved UMAP cache: {emb.shape}")

emb_labels = labels
emb_wallets = wallets

**Optional fallback — subsample for speed.** Uncomment to run UMAP on a sample instead. Cluster labels stay on full data; only the 2D plot uses the sample.

In [ ]:
# rng = np.random.default_rng(42)
# SAMPLE = 10_000
# idx = rng.choice(len(X), SAMPLE, replace=False)
# emb = umap_embed(X[idx])
# emb_labels = labels[idx]
# emb_wallets = wallets[idx]
# print(f"UMAP shape: {emb.shape} (sampled from {len(X):,})")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))
palette = sns.color_palette("tab10", n_colors=K)

for c in sorted(set(emb_labels)):
    mask = emb_labels == c
    ax.scatter(
        emb[mask, 0], emb[mask, 1],
        s=6, alpha=0.4, color=palette[c],
        label=f"{c}: {ARCHETYPES[c]}",
    )

ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
sample_note = f"sampled from {len(labels):,}" if len(emb_labels) < len(labels) else f"n={len(emb_labels):,}"
ax.set_title(f"ChainSense wallet clusters (k={K}, {sample_note})")
ax.legend(markerscale=3, loc="best", framealpha=0.9)
plt.tight_layout()
plt.savefig(DOCS / "umap_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Etherscan validation

Pick wallets from each cluster and look them up on etherscan.io. Behavior should match the archetype. If three of five "whales" are random one-tx wallets, your cluster doesn't mean what you named it.

**This is the test that matters most.** Spend real time here — 15-20 minutes.

In [ ]:
# Top-N by activity
top = top_wallets_per_cluster(wallets, labels, features, sort_by="tx_count", n=5)
for c in sorted(top["cluster"].unique()):
    print(f"\nCluster {c} ({ARCHETYPES[c]}) — top 5 by tx_count:")
    for _, row in top[top["cluster"] == c].iterrows():
        print(f"  https://etherscan.io/address/{row['wallet']}  (tx_count={row['tx_count']:.0f})")

In [ ]:
# Random sample — catches case where top wallets are unrepresentative
print("Random sample per cluster:\n")
clusters_df = pd.DataFrame({"wallet": wallets, "cluster": labels})
for c in sorted(set(labels)):
    in_c = clusters_df[clusters_df["cluster"] == c]
    sample = in_c.sample(min(3, len(in_c)), random_state=42)
    print(f"Cluster {c} ({ARCHETYPES[c]}):")
    for w in sample["wallet"]:
        print(f"  https://etherscan.io/address/{w}")
    print()

## 8. Outlier detection

IsolationForest flags the weirdest ~1% of wallets. Often drainers, sanctioned addresses, MEV bots, or fresh exploit contracts. Cheap to compute, high portfolio value — recruiters like a project that surfaces concrete weird things.

In [ ]:
iso_model, scores, is_outlier = fit_isolation_forest(X, contamination=0.01)

outlier_wallets = wallets[is_outlier]
outlier_scores = scores[is_outlier]
order = np.argsort(outlier_scores)   # lowest = most anomalous

print(f"\nTop 10 most anomalous wallets (investigate on Etherscan):")
for i in order[:10]:
    print(f"  score={outlier_scores[i]:>6.3f}  https://etherscan.io/address/{outlier_wallets[i]}")

## 9. Save final assignments

`clusters.parquet` is the canonical output. UMAP coords go in only if the embedding covers all wallets; if you used the subsample fallback, those columns are skipped.

In [ ]:
result_cols = {
    "wallet": wallets,
    "cluster": labels,
    "archetype": [ARCHETYPES[l] for l in labels],
    "is_outlier": is_outlier,
    "anomaly_score": scores,
    "pca_x": pca_emb[:, 0],
    "pca_y": pca_emb[:, 1],
}

# Attach UMAP coords only if embedding is full-size
if len(emb) == len(wallets):
    result_cols["umap_x"] = emb[:, 0]
    result_cols["umap_y"] = emb[:, 1]
else:
    print(f"Skipping UMAP coords (embedding is {len(emb):,}, wallets is {len(wallets):,})")

results = pd.DataFrame(result_cols)
results.to_parquet(PROCESSED / "clusters.parquet", index=False)
print(f"Saved {len(results):,} wallet assignments to clusters.parquet")
results.head()

## 10. Findings

Fill this in after validation. This text becomes the README writeup.

### Summary
- N wallets clustered into K archetypes over [block range]
- Best silhouette: ___ at k=___
- Largest cluster: ___ (__%)

### Archetypes
- **Cluster 0 (name)**: characterized by ___. Example wallets: 0x..., 0x...
- **Cluster 1 (name)**: ___
- **Cluster 2 (name)**: ___
- _continue for each cluster_

### Outliers
- N wallets flagged anomalous
- Notable findings from manual Etherscan investigation: ___

### Limitations
- ___ wasn't captured by current features
- Cluster boundaries are fuzzy where ___
- Window is short (X blocks) — longer-timescale patterns not visible

### Next steps
- Anomaly deep-dive on flagged wallets
- Algorithmic trading strategy using cluster signals (Week 8)
- Dashboard with live cluster assignment for new wallets